# AgentOps Lab 02 - Agent or workflow?

This notebook teaches one of the most important agent engineering decisions: do you actually need an agent?

You will solve three checkout operations tasks for the same fictional SaaS company. The tasks look similar on the surface, but they deserve different architectures. The lesson is to choose the least autonomous design that reliably solves the problem.



## Notebook-first learning contract

This notebook is the primary lesson for this topic. The Python module is not a separate replacement for the lesson; it is the implementation layer that the notebook explains, runs, breaks, and evaluates. Work through the notebook in this order:

1. read the concept model and architecture boundary;
2. inspect the tool/state/policy contracts;
3. run the deterministic implementation;
4. trigger the deliberate failure case;
5. record evaluation, cost, latency, and safety observations; and
6. answer the architecture question before moving on.


## Deep-dive training guide — Agent or workflow decision

### Concepts to master

- architecture ladder: deterministic workflow, bounded workflow, bounded agent, multi-agent team
- problem shape as the driver of autonomy
- dynamic evidence path versus known steps

### Implementation walkthrough

Study how Task A, Task B, and Task C differ in `workflow_or_agent.py`. The code is intentionally small so you can see exactly where control moves from deterministic code to model-directed investigation.

### Deliberate failure case

Force Task C through the deterministic Task A workflow. The output will be cheap and fast, but it will miss deployment/log evidence and produce a weak recommendation.

### Learner exercise

Add Task D: `VIP customers report intermittent invoice checkout failures`. Decide whether it is a workflow, agentic workflow, single agent, or team, then justify your route with expected evidence.

### What to write down

For each run, capture the chosen architecture, tool trajectory, evidence used, rejected alternatives, stop condition, estimated cost, latency, and one sentence explaining whether the architecture was the least autonomous reliable option.


## Engineering checklist for this notebook

Use this checklist as your mini design review before you call the topic complete.

| Area | Question to answer |
| --- | --- |
| Control boundary | Which decisions are made by deterministic code, and which are delegated to the model? |
| Tools | Are tool inputs typed, narrow, authorized, and auditable? |
| State | What state is carried between steps, and what should never become long-term memory? |
| Failure mode | What is the easiest way this design loops, overacts, or fabricates certainty? |
| Evaluation | Which outcome, trajectory, safety, cost, and latency signals prove the design is working? |
| Architecture choice | Why is this architecture simpler or better than the nearest alternative? |


## Architecture ladder

The article's core design ladder is practical rather than philosophical:

| Problem shape | Architecture |
| --- | --- |
| Known steps | Workflow |
| Few model decisions | Agentic workflow |
| Dynamic investigation | Agent |
| Specialized separable work | Multi-agent |

More autonomy is not automatically better. It usually adds more states, more failure paths, more cost, more evaluation work, and more things to debug. The useful question is: what is the shortest reliable trajectory to a correct result?


```mermaid
flowchart LR
    A["Known steps"] --> B["Deterministic workflow"]
    C["Known path with limited judgment"] --> D["Bounded agentic workflow"]
    E["Evidence path discovered at runtime"] --> F["Single bounded agent"]
    G["Separable specialist roles"] --> H["Multi-agent team"]
```


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root / "labs"))

from agentops_lab.workflow_or_agent import (
    classify_task,
    get_recent_deployments,
    query_region_logs,
    task_a_status_report,
    task_b_bounded_workflow,
    task_c_dynamic_investigation,
)


## Task A - Status report

**Request:** Retrieve the current status of checkout and format a status report.

Expected architecture:

```mermaid
flowchart TD
    S["get_service_status(checkout)"] --> R["format status report"]
```

No agent is required. The path is fully known before runtime. A model-controlled loop would add latency, cost, and debugging surface without changing the answer.


In [ ]:
task_a = task_a_status_report()
print(task_a.architecture)
print(task_a.reason)
print("steps:", " -> ".join(task_a.steps))
print(task_a.result)


## Task B - Bounded workflow

**Request:** Check whether checkout is unhealthy. If unhealthy, retrieve the runbook and summarize the recommended response.

This is still not a fully open-ended agent. The application knows the control path. A model may help summarize the runbook, but code should own the branch and stop condition.

```mermaid
flowchart TD
    A["Get checkout status"] --> B{"Healthy?"}
    B -- "Yes" --> C["Finish: no runbook needed"]
    B -- "No" --> D["Retrieve checkout runbook"]
    D --> E["Summarize recommended response"]
```


In [ ]:
task_b = task_b_bounded_workflow()
print(task_b.architecture)
print(task_b.reason)
print("steps:", " -> ".join(task_b.steps))
print(task_b.result)


## Task C - Dynamic investigation

**Request:** Investigate reports that some European customers cannot complete checkout.

Now the path is not obvious. The system may need to inspect service health, incidents, deployments, regional logs, and runbooks. Which tool matters next depends on what the previous observation says. That is the first task here that justifies a bounded agent.

```mermaid
flowchart TD
    G["Goal: European checkout failures"] --> H["Inspect service health"]
    H --> I["Query active incidents"]
    I --> J["Inspect recent deployments"]
    J --> K["Query regional logs"]
    K --> L["Retrieve runbook"]
    L --> M["Recommend response with caveats"]
```

Even here, the agent should be bounded. The application still owns max steps, available tools, evidence formatting, and the rule that correlation is not root cause.


In [ ]:
task_c = task_c_dynamic_investigation()
print(task_c.architecture)
print(task_c.reason)
print("steps:", " -> ".join(task_c.steps))
print(task_c.result)


In [ ]:
for item in task_c.evidence:
    print("\nTOOL:", item["tool"])
    print("ARGS:", item["arguments"])
    print("RESULT:", item["result"])


## Classification practice

Use this tiny classifier as a thinking aid, not as a production rule engine. The real skill is explaining why the architecture matches the problem shape.


In [ ]:
for problem in [
    "Known steps: read status and format a report",
    "Few model decisions: branch on health and summarize a runbook",
    "Dynamic investigation: decide which evidence to inspect next",
    "Specialized separable work: logs analyst, deployment analyst, support writer",
]:
    print(f"{problem} -> {classify_task(problem)}")


## Compare the trajectories

A useful architecture choice should show up in the trajectory. Task A has two predictable steps. Task B adds one branch. Task C has more evidence collection because the system must discover what matters at runtime.


In [ ]:
runs = [task_a, task_b, task_c]
for run in runs:
    print(f"{run.task}\n  architecture: {run.architecture}\n  step_count: {len(run.steps)}\n  steps: {' -> '.join(run.steps)}\n")


## Experiments

- Change Task B so checkout is healthy. Does the workflow stop before retrieving the runbook?
- Add a `query_customer_tickets(region)` tool to Task C. When should the agent call it?
- Split Task C into deployment analyst, logs analyst, and support response writer. What evidence would justify the extra coordination cost?
- Define an evaluation dataset with examples for each architecture class. What would make a classifier overuse agents?

References: [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/), [Anthropic Building effective agents](https://www.anthropic.com/engineering/building-effective-agents), and [OpenAI's practical guide to building AI agents](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/).
